# Differential Competition Report for Throughput - Internal use only DO NOT SHARE

This informs ISPs about suboptimal routing or interconnects, where
their content is taking a slower path (lower throughput) to reach some destinations.

ISPs are likely to be extremely sensitve about exposing this data
to their competition.

IMPORTANT: To share the data, export the spreadsheet below as a csv, **not a link to the dashboard itself**.
(Click the top right dropdown, click inspect, click download csv.)

Also share the [documentation](https://docs.google.com/document/d/1H6cd2Y2W4ljw-Uw2ADvg8xQSbOjjISjxFUIgaMwL0Qk/edit?usp=sharing)
for Competiton Reports.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "method",
    "type": "custom",
    "label": "",
    "description": "",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "cached", "value": "cached" },
      { "text": "live", "value": "live" }
    ],
    "current": { "value": "cached" },
    "query_sql": "cached, live, experimental, exp-show, exp-showAll, exp-ndt5ok, exp-showIPv"
  },
  {
    "name": "organization",
    "type": "query",
    "label": "Organization Overrides",
    "description": "Manual organization selector, to access non-automatically detected organizations. ",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "All orgs", "value": ".*" }
    ],
    "current": { "value": ".*" },
    "query_sql": "SELECT text, value FROM ( SELECT 'All orgs' AS text, '.*' AS value, 0 AS _sort UNION ALL SELECT DISTINCT  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS text,  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS value,  1 AS _sort FROM `mlab-collaboration.mm_preproduction.cached_metadata` WHERE REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') IS NOT NULL) ORDER BY _sort, text"
  },
  {
    "name": "radius",
    "type": "custom",
    "label": "Radius (kM)",
    "description": "How far should servers be included?",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "100", "value": "100" },
      { "text": "500", "value": "500" }
    ],
    "current": { "value": "100" },
    "query_sql": "1, 100, 500"
  },
  {
    "name": "ISPcount",
    "type": "custom",
    "label": "Client ISP count",
    "description": "Number of client ISPs to evaluate.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "5", "value": "5" },
      { "text": "10", "value": "10" },
      { "text": "20", "value": "20" },
      { "text": "50", "value": "50" }
    ],
    "current": { "value": "5" },
    "query_sql": "5,10,20, 50"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---
ctrl = Controls(VARIABLES, client, presets=url_params)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")

# End date + duration — ignored when method=cached.
# End defaults to the most recent Sunday (UTC); duration defaults to 7 days.
_today_utc  = datetime.now(timezone.utc).date()
_days_back  = (_today_utc.weekday() + 1) % 7
_end_date   = _today_utc - timedelta(days=_days_back)
w_to       = widgets.DatePicker(value=_end_date, description='End (UTC)',
                                style={"description_width": "90px"})
w_duration = widgets.Dropdown(
    options=[("1 day", 1), ("7 days", 7), ("30 days", 30)],
    value=7, description='Duration',
    style={"description_width": "90px"},
)
w_from = None
_date_label = widgets.HTML('')   # filled from query results after Run


In [ ]:
# --- Competition report panels ---
_DATASET     = "mlab-collaboration.mm_preproduction"
_REPORT_TYPE = "throughput"   # minRTT or throughput
_X_AXIS      = "none"
_BIN_SIZE    = 50

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    method  = ctx.get("method", "cached")
    to_dt   = (datetime.combine(w_to.value, time(), tzinfo=timezone.utc)
               if w_to and w_to.value else datetime.now(timezone.utc))
    from_dt = to_dt - timedelta(days=w_duration.value if w_duration else 7)

    out.clear_output(wait=True)
    with out:
        if method == "cached":
            _date_label.value = (
                '<div style="font-size:12px;color:grey;margin:2px 0"><b>Cached data:</b> '
                + rt.get_cached_date_range(client, _DATASET) + '</div>')
        try:
            df = rt.run_competition_report(
                client, _REPORT_TYPE, method,
                ctx.get("organization", ".*"),
                int(ctx.get("radius", 100)),
                int(ctx.get("ISPcount", 5)),
                from_dt, to_dt,
                _DATASET,
            )
        except Exception as exc:
            display(HTML(f"<pre>query failed: {exc}</pre>"))
            df = None

        if df is not None and not df.empty:
            display(HTML(
                '<div style="height:600px;overflow:auto">'
                + df.to_html(index=False, na_rep="")
                + '</div>'
            ))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, "Selector Diagnostics")
        _diag_acc.selected_index = None
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
if w_from is not None:
    _date_row = widgets.HBox([w_from, w_to], layout=widgets.Layout(margin='2px 0'))
elif w_to is not None and w_duration is not None:
    _date_row = widgets.HBox([w_to, w_duration], layout=widgets.Layout(margin='2px 0'))
else:
    _date_row = widgets.HTML('')
display(widgets.VBox([ctrl.box, _date_label, _date_row, w_run, out]))
